In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from sklearn.feature_selection import SelectKBest, f_classif
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC, NuSVC
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from IPython.display import display

# Chargement des données
data = pd.read_csv('parkinson.csv')

# Préparation des données
X = data.drop(['ID', 'Recording', 'Status'], axis=1)
y = data['Status']

# Conversion de la variable 'Gender'
X = pd.get_dummies(X, columns=['Gender'], drop_first=True)

# Normalisation
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Sélection des caractéristiques (top 20)
selector = SelectKBest(score_func=f_classif, k=20)
X_selected = selector.fit_transform(X_scaled, y)

# Classifieurs
classifiers = {
    "Naïve Bayes": GaussianNB(),
    "SVM (C-SVM)": SVC(kernel='rbf', C=1.0, random_state=42),
    "SVM (nu-SVM)": NuSVC(kernel='rbf', nu=0.5, random_state=42),
    "MLP": MLPClassifier(hidden_layer_sizes=(100,), max_iter=1000, random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42)
}

# Fonction d'évaluation avec cross-validation
def evaluate_model_cv(model, X, y, cv=10):
    return {
        "Accuracy": cross_val_score(model, X, y, cv=cv, scoring='accuracy').mean(),
        "Precision": cross_val_score(model, X, y, cv=cv, scoring='precision').mean(),
        "Recall": cross_val_score(model, X, y, cv=cv, scoring='recall').mean(),
        "F1-score": cross_val_score(model, X, y, cv=cv, scoring='f1').mean()
    }

# Évaluation sans feature selection
results_no_fs = {}
for name, clf in classifiers.items():
    scores = evaluate_model_cv(clf, X_scaled, y)
    results_no_fs[name] = scores


# Affichage résultats
print("\n🎯 Résultats sans sélection de caractéristiques :")
results_df_no_fs = pd.DataFrame(results_no_fs).T
display(results_df_no_fs.round(4))




In [ ]:
def plot_results(df, title):
    df.plot(kind='bar', figsize=(12, 6), colormap='viridis')
    plt.title(title)
    plt.ylabel("Score")
    plt.ylim(0, 1.1)
    plt.grid(axis='y')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

plot_results(results_df_no_fs, "Performance sans sélection de caractéristiques (10-fold CV)")


In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split

# Refaire un split (car la validation croisée ne donne pas de prédictions)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, stratify=y, test_size=0.2, random_state=42)

# Affichage des matrices de confusion pour chaque modèle
for name, clf in classifiers.items():
    # Entraînement
    clf.fit(X_train, y_train)
    
    # Prédiction
    y_pred = clf.predict(X_test)
    
    # Matrice de confusion
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=clf.classes_)
    
    # Affichage
    print(f"🔍 Matrice de confusion - {name}")
    disp.plot(cmap='Blues')
    plt.title(f"Matrice de confusion - {name}")
    plt.show()
